# **IBOVESPA Trend Prediction**

## Exploratory Data Analysis (EDA)

### Objetivo

O objetivo desta análise exploratória é compreender o comportamento histórico do índice **IBOVESPA**, identificar padrões nos dados e preparar as variáveis para a construção de um modelo preditivo.

A análise busca:

- Entender a distribuição das variáveis do dataset  
- Identificar possíveis inconsistências ou valores ausentes  
- Explorar relações entre preço de abertura, fechamento, máxima, mínima e volume  
- Avaliar padrões temporais do índice  
- Preparar os dados para a etapa de **modelagem preditiva**, cujo objetivo é prever se o **fechamento do IBOVESPA no dia seguinte será maior ou menor que o do dia atual**.

## **1. Import Lab**

In [106]:
# Manipulação e cálculos
import pandas as pd #Ler, limpar, transformar tabelas
import numpy as np #Cálculos numéricos rápidos (base para Pandas)

# Visualização
import matplotlib.pyplot as plt #Criação de gráficos personalizados
import matplotlib.ticker as mtick #formatar, posicionar e controlar os rótulos (ticks) dos eixos X e Y dos gráficos.

# Utilidades
import unicodedata
from pathlib import Path

## **2. Personalizar as configurações de exibição do Pandas**

In [107]:
pd.set_option('display.max_columns', None) #Exibir todas as colunas
pd.set_option('display.max_rows', None) #Exibir todas as linhas

## **Base de Dados**
**Dados:** https://br.investing.com/indices/bovespa-historical-data<br>
**Filtro:** 01/01/2024 a 04/03/2026

## **3. Carregar os Arquivo .csv**

In [108]:
RAW = Path("../data/raw")

df_dados_ibovespa = pd.read_csv(
    RAW / "ibov_dados_historicos_012024_032026.csv", 
    sep=","
)

## **4. Visualização e Informações do Dataset**

In [109]:
# Primeiras linhas
df_dados_ibovespa.head()

,Data,Último,Abertura,Máxima,Mínima,Vol.,Var%
0,04.03.2026,185.770,183.110,186.306,183.110,"7,38M","1,46%"
1,03.03.2026,183.105,189.284,189.602,180.518,"14,37B","-3,28%"
2,02.03.2026,189.307,188.786,190.110,186.638,"9,09B","0,28%"
3,27.02.2026,188.787,191.005,191.005,188.478,"11,00B","-1,16%"
4,26.02.2026,191.005,191.248,191.978,188.977,"8,80B","-0,13%"


In [110]:
# Últimas linhas
df_dados_ibovespa.tail()

,Data,Último,Abertura,Máxima,Mínima,Vol.,Var%
538,08.01.2024,132.427,132.023,132.498,131.015,"8,50M","0,31%"
539,05.01.2024,132.023,131.218,132.635,130.579,"9,20M","0,61%"
540,04.01.2024,131.226,132.831,132.885,131.024,"8,97M","-1,21%"
541,03.01.2024,132.834,132.697,133.576,132.250,"8,70M","0,10%"
542,02.01.2024,132.697,134.186,134.195,132.095,"8,44M","-1,11%"


In [111]:
# Tamanho do dataframe
df_dados_ibovespa.shape

(543, 7)

In [112]:
# Informações gerais (tipos e nulos)
df_dados_ibovespa.info()

<class 'pandas.DataFrame'>
RangeIndex: 543 entries, 0 to 542
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Data      543 non-null    str    
 1   Último    543 non-null    float64
 2   Abertura  543 non-null    float64
 3   Máxima    543 non-null    float64
 4   Mínima    543 non-null    float64
 5   Vol.      543 non-null    str    
 6   Var%      543 non-null    str    
dtypes: float64(4), str(3)
memory usage: 29.8 KB


In [113]:
# Estatísticas descritivas (numéricas)
df_dados_ibovespa.describe()

,Último,Abertura,Máxima,Mínima
count,543.000000,543.000000,543.000000,543.000000
mean,136.635144,136.540654,137.456223,135.750748
std,15.459281,15.315560,15.567554,15.171848
min,118.533000,118.534000,119.451000,118.223000
25%,127.312500,127.311000,127.973000,126.475000
50%,131.447000,131.447000,132.016000,130.579000
75%,139.318500,139.279500,140.018500,138.511500
max,191.490000,191.491000,192.624000,190.419000


In [114]:
# Quantidade de nulos por coluna
df_dados_ibovespa.isna().sum()

Data        0
Último      0
Abertura    0
Máxima      0
Mínima      0
Vol.        0
Var%        0
dtype: int64

In [115]:
# Checar duplicados
df_dados_ibovespa.duplicated().sum()

np.int64(0)

## **5. Tratamento da Base**
Preparar o dataset para análise e modelagem:
1. Converter tipos de variáveis (Data, Vol., Var%)
2. Garantir ordem cronológica crescente
3. Definir `Data` como índice (séries temporais)

### **5.1. Converter os tipos de variáveis**
- **Data:** `object` → `datetime`
- **Vol.:** `object` → `float` (tratando sufixos **K**, **M**, **B**)
- **Var%:** `object` → `float` (removendo `%` e ajustando decimal)

### **5.1.1. Data (object → datetime)**

In [116]:
# Data: object → datetime
df_dados_ibovespa["Data"] = pd.to_datetime(
    df_dados_ibovespa["Data"],
    format="%d.%m.%Y",
    errors="coerce"  # transforma inválidos em NaT ao invés de quebrar
)

### **5.1.2 Vol. (object → float)**

In [117]:
# Vol.: object → float
# Trata K (mil), M (milhões), B (bilhões), além de '-', vazio e NaN
def tratar_volume(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip()

    if valor in ("", "-", "—"):
        return np.nan

    valor = valor.replace(",", ".")  # ajusta decimal

    multipliers = {"K": 1_000, "M": 1_000_000, "B": 1_000_000_000}

    sufixo = valor[-1]
    if sufixo in multipliers:
        return float(valor[:-1]) * multipliers[sufixo]

    # se vier número "puro"
    return float(valor)

df_dados_ibovespa["Vol."] = df_dados_ibovespa["Vol."].apply(tratar_volume)

### **5.1.3 Var% (object → float)**

In [118]:
# Var%: object → float
# Remove % e ajusta vírgula decimal
df_dados_ibovespa["Var%"] = (
    df_dados_ibovespa["Var%"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.replace(",", ".", regex=False)
    .replace({"-": np.nan, "—": np.nan, "": np.nan})
    .astype(float)
)

### **5.2. Variável Data: Ordem cronológica crescente**
Como os cálculos consideram a linha anterior como “dia anterior”, os dados precisam estar
do mais antigo para o mais recente para manter a sequência temporal correta.

In [119]:
# Verificar se já está em ordem crescente
df_dados_ibovespa["Data"].is_monotonic_increasing

False

In [120]:
# Ordenar pela Data (crescente) e resetar índice
df_dados_ibovespa = (
    df_dados_ibovespa
    .sort_values("Data", ascending=True)
    .reset_index(drop=True)
)

### **5.3. Adicionar Data como índice**
Definir a coluna `Data` como índice facilita operações de séries temporais
(resample, rolling, etc.).

In [121]:
df_dados_ibovespa = df_dados_ibovespa.set_index("Data")

### **6. Criar uma coluna com o retorno percentual diário**
O retorno percentual diário mede a variação do preço de fechamento em relação ao dia anterior.

Essa métrica é amplamente utilizada em análise financeira para avaliar o desempenho do ativo ao longo do tempo.

A fórmula do retorno é:

retorno = (preço atual / preço anterior) - 1

In [122]:
#Retorno em decimal
df_dados_ibovespa["retorno_diario"] = df_dados_ibovespa["Último"].pct_change()

#Retorno em percentual (%)
df_dados_ibovespa["retorno_diario%"] = (
    df_dados_ibovespa["Último"].pct_change() * 100
)

In [123]:
df_dados_ibovespa.head ()

,Último,Abertura,Máxima,Mínima,Vol.,Var%,retorno_diario,retorno_diario%
Data,,,,,,,,
2024-01-02,132.697,134.186,134.195,132.095,8440000.0,-1.11,NaN,NaN
2024-01-03,132.834,132.697,133.576,132.250,8700000.0,0.10,0.001032,0.103243
2024-01-04,131.226,132.831,132.885,131.024,8970000.0,-1.21,-0.012105,-1.210533
2024-01-05,132.023,131.218,132.635,130.579,9200000.0,0.61,0.006073,0.607349
2024-01-08,132.427,132.023,132.498,131.015,8500000.0,0.31,0.003060,0.306007


### **7. Salvando os dados tratados**
Após realizar o tratamento e limpeza da base, salvamos o dataset processado na pasta **`data/processed`**.

Separar os dados em **raw** e **processed** é uma boa prática em projetos de dados, pois:

- **raw:** mantém os dados originais sem alteração  
- **processed:** contém os dados já tratados e prontos para análise ou modelagem  

Isso facilita a reprodutibilidade do projeto e evita modificar os dados originais.

In [124]:
# Caminho da pasta
PROCESSED = Path("../data/processed")

# Salvar dataset tratado
df_dados_ibovespa.to_csv(
    PROCESSED / "ibov_dados_tratados.csv"
)